# Outcome-map RV — autopsy

The league named a best row. This notebook tries to break it: chronological
halves, the one-parameter-at-a-time neighbourhood, the two placebo worlds
(Gaussian tree, wrong calendar) re-derived from atoms up, the trade log, and
the five pre-declared kill criteria answered one at a time.

In [1]:
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append("../../")
sys.path.append(".")
from linvol_grid_common import pick_winner                  # noqa: E402
from RVUtils.SFRRVLab.stats import (                        # noqa: E402
    deflated_for_grid, neighbourhood_stability, nw_tstat)

DATA = Path("../data/outcome_map")
league = pd.read_parquet(DATA / "league.parquet")
real = league[league["world"] == "real"].reset_index(drop=True)
trades = pd.read_parquet(DATA / "trades_real.parquet")
with open(DATA / "dailies_real.pkl", "rb") as fh:
    dailies = pickle.load(fh)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
live = real[real["n_trades"] > 0]
w = pick_winner(live)
i = int(w.name)
tr = trades[trades["config"] == i].sort_values("entry").reset_index(drop=True)
d = dailies.get(i, pd.Series(dtype=float))
print(f"winner: {w['expression']} | dte {w['dte']} | thr {w['thr_pp']}pp | "
      f"exit {w['exit']} | linear {w['linear']} | {w['direction']}")
print(f"{int(w['n_trades'])} trades, gross {w['gross_bp']:+.1f}bp, "
      f"net {w['net_1x_bp']:+.1f} @1x / {w['net_2x_bp']:+.1f} @2x")

winner: pair_odd | dte 0-130 | thr 16.0pp | exit hold | linear zq | fade
38 trades, gross +143.8bp, net -1.1 @1x / -146.1 @2x


## 1. Chronological halves

The single most useful robustness check in a one-cycle sample: does the second
half of the history look anything like the first?

In [2]:
if len(tr):
    mid = tr["entry"].quantile(0.5)
    rows = []
    for label, m in (("first half", tr["entry"] <= mid),
                     ("second half", tr["entry"] > mid)):
        g = tr[m]
        rows.append({"half": label, "n": len(g),
                     "from": str(g["entry"].min().date()) if len(g) else "-",
                     "to": str(g["entry"].max().date()) if len(g) else "-",
                     "gross_bp": round(float(g["gross_bp"].sum()), 1),
                     "net_1x_bp": round(float(g["net_1x_bp"].sum()), 1),
                     "per_trade_gross": round(float(g["gross_bp"].mean()), 3),
                     "hit_1x": round(float((g["net_1x_bp"] > 0).mean()), 3)})
    print(pd.DataFrame(rows).to_string(index=False))

       half  n       from         to  gross_bp  net_1x_bp  per_trade_gross  hit_1x
 first half 19 2022-02-16 2024-03-13     113.9       41.1            5.996   0.474
second half 19 2024-04-11 2026-05-05      29.8      -42.3            1.570   0.211


In [3]:
print("=== by calendar year (gross, so the cost model cannot mask a regime) ===")
if len(tr):
    yr = tr.groupby(tr["entry"].dt.year).agg(
        n=("gross_bp", "size"), gross_bp=("gross_bp", "sum"),
        per_trade=("gross_bp", "mean"), net_1x=("net_1x_bp", "sum"))
    print(yr.round(2).to_string())

=== by calendar year (gross, so the cost model cannot mask a regime) ===
        n  gross_bp  per_trade  net_1x
entry                                 
2022    7     46.64       6.66   22.46
2023   10     64.55       6.45   24.71
2024    8     10.60       1.32  -19.47
2025   10     17.12       1.71  -21.24
2026    3      4.86       1.62   -7.61


## 2. Neighbourhood

A knife-edge optimum is visible as a knife edge. Every row below holds the
winner's parameters fixed except one.

In [4]:
params = ["expression", "dte", "thr_pp", "exit", "linear", "direction"]
nb = neighbourhood_stability(live, w, params, metric="net_1x_bp")
print(nb.to_string(index=False))
pos = int((nb["net_1x_bp"] > 0).sum())
print(f"\nneighbourhood: {pos}/{len(nb)} positive at 1x, "
      f"median {nb['net_1x_bp'].median():+.1f}bp")

     param        value   net_1x_bp  n_trades  is_best
expression     map_full   -8.284815        12    False
expression     pair_odd   -1.148236        38     True
expression pair_odd_dev -122.149569        35    False
expression     pair_raw  -87.015845        41    False
expression       reswin  -14.916697        14    False
       dte        0-130   -1.148236        38     True
       dte    130-10000 -142.963387        54    False
    thr_pp          4.0 -114.439397        46    False
    thr_pp          8.0  -69.754359        46    False
    thr_pp         16.0   -1.148236        38     True
      exit     converge -109.053533        48    False
      exit         hold   -1.148236        38     True
    linear         none  -42.024707        38    False
    linear         swap  -55.756427        38    False
    linear           zq   -1.148236        38     True
 direction         fade   -1.148236        38     True
 direction     momentum -288.662083        38    False

neighbour

## 3. The placebos

**P1 Gaussian tree** — the fair value is a moment-matched Gaussian at the
forward and the cells are placed on a lattice-free 25bp grid, so neither the
prices nor the atom locations borrow anything from the FOMC lattice; only the
map's size (a calendar fact) is kept.
**P2 wrong calendar** — every meeting wears the next meeting's jump, support
and mantissa, and the atoms, cells, fair values and hedge ratios are all
re-derived inside that world.

If the edge survives either, it was never lattice information.

In [5]:
rows = []
for world, g in league.groupby("world"):
    gl = g[g["n_trades"] > 0]
    if gl.empty:
        continue
    gw = pick_winner(gl)
    rows.append({
        "world": world, "configs": len(g), "with_trades": len(gl),
        "best_gross": round(float(gl["gross_bp"].max()), 1),
        "best_net_1x": round(float(gl["net_1x_bp"].max()), 1),
        "median_gross": round(float(gl["gross_bp"].median()), 1),
        "winner_expr": gw["expression"], "winner_n": int(gw["n_trades"]),
        "winner_gross": round(float(gw["gross_bp"]), 1),
    })
pl = pd.DataFrame(rows).set_index("world")
print(pl.to_string())

             configs  with_trades  best_gross  best_net_1x  median_gross winner_expr  winner_n  winner_gross
world                                                                                                       
p1_gauss         360          336       108.4          8.3           0.0    map_full        12          49.0
p2_calendar      360          360       162.9          5.0           0.0    map_full        17          25.9
real             360          360       143.8         -1.1           0.0    pair_odd        38         143.8


In [6]:
print("=== the winner's OWN coordinates, in each world ===")
coords = {k: w[k] for k in params}
rows = []
for world, g in league.groupby("world"):
    m = np.ones(len(g), dtype=bool)
    for k, v in coords.items():
        m &= (g[k] == v).to_numpy()
    if not m.any():
        rows.append({"world": world, "found": False})
        continue
    r = g[m].iloc[0]
    rows.append({"world": world, "found": True, "n": int(r["n_trades"]),
                 "gross_bp": round(float(r["gross_bp"]), 1),
                 "net_1x_bp": round(float(r["net_1x_bp"]), 1),
                 "per_trade_gross": round(
                     float(r["gross_bp"] / max(r["n_trades"], 1)), 3)})
same = pd.DataFrame(rows)
print(same.to_string(index=False))
if len(same) > 1 and same["found"].all():
    base = float(same[same["world"] == "real"]["gross_bp"].iloc[0])
    for _, r in same[same["world"] != "real"].iterrows():
        keep = float(r["gross_bp"]) / base if base else np.nan
        print(f"  {r['world']}: retains {keep:.0%} of the real gross")
    print("\nA placebo that retains most of the gross means the signal was "
          "never lattice information — the pre-declared kill criterion 3.")

=== the winner's OWN coordinates, in each world ===
      world  found  n  gross_bp  net_1x_bp  per_trade_gross
   p1_gauss   True 25      30.2      -60.9            1.209
p2_calendar   True 37     126.6       -4.1            3.421
       real   True 38     143.8       -1.1            3.783
  p1_gauss: retains 21% of the real gross
  p2_calendar: retains 88% of the real gross

A placebo that retains most of the gross means the signal was never lattice information — the pre-declared kill criterion 3.


## 4. The trade log

Individual trades, so a single episode carrying the result is visible as a
single episode.

In [7]:
if len(tr):
    # measured against GROSS: this winner's net is ~0, so a share-of-net
    # ratio would be a meaningless large number
    conc = float(tr["gross_bp"].abs().max() / max(abs(tr["gross_bp"].sum()),
                                                  1e-9))
    top3 = float(tr["gross_bp"].nlargest(3).sum() / max(tr["gross_bp"].sum(),
                                                        1e-9))
    print(f"largest single trade is {conc:.0%} of total gross; "
          f"the top 3 are {top3:.0%} of it")
    show = tr[["symbol", "entry", "exit", "side", "n_contracts",
               "entry_signal_bp", "exit_signal_bp", "opt_gross_bp",
               "hedge_bp", "opt_cost_bp", "lin_cost_bp", "net_1x_bp",
               "exit_reason"]]
    print(show.head(15).round(3).to_string(index=False))
    print("\n=== exit reasons ===")
    print(tr["exit_reason"].value_counts().to_string())
    print("\n=== signal decay: did the thing we entered on actually converge? "
          "===")
    conv = (tr["exit_signal_bp"].abs() / tr["entry_signal_bp"].abs())
    print(f"  |exit signal| / |entry signal|: median {conv.median():.2f}, "
          f"share below 1: {(conv < 1).mean():.1%}")
    print(f"  mean holding: {(tr['exit'] - tr['entry']).dt.days.mean():.1f} "
          f"calendar days")

largest single trade is 20% of total gross; the top 3 are 55% of it
symbol      entry       exit  side  n_contracts  entry_signal_bp  exit_signal_bp  opt_gross_bp  hedge_bp  opt_cost_bp  lin_cost_bp  net_1x_bp exit_reason
SFRM22 2022-02-16 2022-03-10     1          6.0           -1.670           2.788         2.496     2.593          1.5        2.313      1.275   time_stop
SFRU22 2022-05-12 2022-06-03     1          8.0            1.399          -4.917        -0.147    -2.365          2.0        1.614     -6.125   time_stop
SFRU22 2022-06-07 2022-06-29     1          8.0           -6.836         -62.753        -3.610    32.585          2.0        1.186     25.789   time_stop
SFRZ22 2022-08-23 2022-09-14     1          8.0           -3.064         -14.574         0.000   -11.808          2.0        0.746    -14.554   time_stop
SFRZ22 2022-09-22 2022-10-13     1          6.0           17.060          35.180         0.898    -2.257          1.5        1.713     -4.573   time_stop
SFRZ22 2

## 5. The linear leg, trade by trade

The hedge is supposed to remove the package's exposure to each meeting's
priced jump. Whether it earns its bill is the whole point of the study.

In [8]:
lin_tr = trades[trades["lin_contracts"] > 0]
if len(lin_tr):
    print(f"trades carrying a linear leg: {len(lin_tr):,}")
    print(f"  hedge P&L   : total {lin_tr['hedge_bp'].sum():+.1f}bp, "
          f"per trade {lin_tr['hedge_bp'].mean():+.3f}bp, "
          f"std {lin_tr['hedge_bp'].std():.3f}")
    print(f"  linear bill : total {lin_tr['lin_cost_bp'].sum():.1f}bp, "
          f"per trade {lin_tr['lin_cost_bp'].mean():.3f}bp")
    print(f"  option bill : per trade {lin_tr['opt_cost_bp'].mean():.3f}bp")
    print(f"  basket size : {lin_tr['lin_contracts'].mean():.2f} contracts "
          f"(3 legs per meeting)")
    print(f"  rebalances  : {lin_tr['n_rebalances'].mean():.2f} per trade")
    # does the hedge reduce dispersion? that is what a hedge is FOR
    j = trades.drop(columns=["expression"]).merge(
        real[["expression", "dte", "thr_pp", "exit", "linear", "direction"]]
        .reset_index().rename(columns={"index": "config"}), on="config")
    piv = j.groupby(["expression", "linear"])["opt_gross_bp"].std()
    tot = j.groupby(["expression", "linear"])["gross_bp"].std()
    print("\n=== std of per-trade P&L: option leg alone vs option+hedge ===")
    print(pd.DataFrame({"opt_only_std": piv, "with_hedge_std": tot,
                        "ratio": (tot / piv)}).round(3).to_string())
    print("\nA hedge that RAISES the standard deviation is adding a position, "
          "not removing risk.")

trades carrying a linear leg: 11,944
  hedge P&L   : total +0.0bp, per trade +0.000bp, std 6.030
  linear bill : total 23024.1bp, per trade 1.928bp
  option bill : per trade 1.758bp
  basket size : 2.10 contracts (3 legs per meeting)
  rebalances  : 0.44 per trade

=== std of per-trade P&L: option leg alone vs option+hedge ===
                     opt_only_std  with_hedge_std  ratio
expression   linear                                     
map_full     none           2.523           2.523  1.000
             swap           2.523           3.196  1.267
             zq             2.523           5.296  2.100
pair_odd     none           2.830           2.830  1.000
             swap           2.830           4.940  1.746
             zq             2.830           7.353  2.598
pair_odd_dev none           2.894           2.894  1.000
             swap           2.894           4.038  1.395
             zq             2.894           5.666  1.958
pair_raw     none           3.082           

### The two-linear-source consistency test

This is the sharpest available check on whether the hedge leg is replicating
anything. The ZQ ladder and the meeting-dated swap ladder measure the *same*
per-meeting jumps and (per the atlas) tie out to a couple of bp. The hedge
ratios are identical — the same `h` vector, the same package, the same trades;
only the market the jump is marked in differs. A hedge that is replicating the
package's meeting exposure must therefore produce nearly the same P&L in both.
If the two disagree materially, the "hedge P&L" is measurement noise wearing a
hedge's clothes, and its sign in this sample is an accident of one rate cycle.

In [9]:
key = ["expression", "dte", "thr_pp", "exit", "direction"]
hp = real.pivot_table(index=key, columns="linear", values="hedge_bp")
if {"zq", "swap"}.issubset(hp.columns):
    both = hp.dropna(subset=["zq", "swap"])
    both = both[(both["zq"].abs() > 1e-9) | (both["swap"].abs() > 1e-9)]
    ratio = (both["swap"] / both["zq"].replace(0, np.nan)).dropna()
    corr = float(both["zq"].corr(both["swap"]))
    print(f"configs with a hedge on both markets: {len(both)}")
    print(f"  correlation of hedge P&L across the two markets : {corr:.3f}")
    print(f"  swap / ZQ hedge P&L ratio: median {ratio.median():.2f}, "
          f"IQR [{ratio.quantile(.25):.2f}, {ratio.quantile(.75):.2f}]")
    print(f"  median |ZQ hedge| {both['zq'].abs().median():.1f}bp vs "
          f"|swap hedge| {both['swap'].abs().median():.1f}bp")
    print("\nTwo markets that agree about the jumps to a couple of bp, marking "
          "the SAME hedge ratios on the SAME trades, should produce the same "
          "hedge P&L. The gap between them is the honest error bar on any "
          "claim that the linear leg contributed.")

configs with a hedge on both markets: 120
  correlation of hedge P&L across the two markets : 0.851
  swap / ZQ hedge P&L ratio: median 0.48, IQR [0.16, 0.70]
  median |ZQ hedge| 40.8bp vs |swap hedge| 23.1bp

Two markets that agree about the jumps to a couple of bp, marking the SAME hedge ratios on the SAME trades, should produce the same hedge P&L. The gap between them is the honest error bar on any claim that the linear leg contributed.


### Is the hedge the right SIZE?

`hedge_bp = -side * sum_m h_m * d(jump_m)`, so `-hedge_bp` is exactly what the
lattice predicted the option package would earn from the meetings repricing.
Regressing the option leg's realised P&L on that prediction, through the
origin, measures the hedge ratio the market actually wants. A beta of 1 means
the frame-frozen ratios are right-sized; a beta well under 1 means the hedge
is scaled to a distribution the market does not use.

The reason to expect a miss here — and the reason the channel-1 study did not
see one — is that a digital's lattice sensitivity is a difference of CDFs
(bounded, smooth) while a butterfly's is a DENSITY. The lattice's density is a
sum of near-atoms; the market's is the same mass smeared by the off-lattice
premium the atlas measures at 15–29pp. For a density-like payoff that
difference is first-order.

In [10]:
cal = trades[(trades["lin_contracts"] > 0) & (trades["hedge_bp"].abs() > 1e-9)]
if len(cal) > 30:
    pred = -cal["hedge_bp"].to_numpy()          # what the lattice predicted
    real_pnl = cal["opt_gross_bp"].to_numpy()   # what the option leg did
    beta = float(np.dot(pred, real_pnl) / np.dot(pred, pred))
    resid = real_pnl - beta * pred
    r2 = 1.0 - float(np.var(resid) / np.var(real_pnl))
    print(f"n = {len(cal):,} hedged trades")
    print(f"  realised option P&L on lattice-predicted P&L: "
          f"beta = {beta:.3f}, R^2 = {r2:.3f}")
    print(f"  std of prediction {pred.std():.2f}bp vs std of realisation "
          f"{real_pnl.std():.2f}bp  (ratio {real_pnl.std() / pred.std():.2f})")
    print(f"\n  -> the frame-frozen ratios are "
          f"{1 / beta:.1f}x too large for this payoff"
          if 0 < beta < 1 else "")

    print("\n=== net P&L if the basket were scaled by k (IN-SAMPLE, post-hoc) "
          "===")
    rows = []
    for k in (0.0, 0.25, beta, 0.5, 1.0):
        net1 = (cal["opt_gross_bp"] + k * cal["hedge_bp"]
                - cal["opt_cost_bp"] - k * cal["lin_cost_bp"])
        rows.append({"k": round(k, 3),
                     "gross_bp": round(float((cal["opt_gross_bp"]
                                              + k * cal["hedge_bp"]).sum()), 1),
                     "net_1x_bp": round(float(net1.sum()), 1),
                     "per_trade_net": round(float(net1.mean()), 3),
                     "std_per_trade": round(float(
                         (cal["opt_gross_bp"] + k * cal["hedge_bp"]).std()), 2)})
    print(pd.DataFrame(rows).to_string(index=False))
    print("\nThis is a post-hoc, in-sample rescaling — it cannot certify a "
          "strategy. It answers one question only: is the linear leg the wrong "
          "SIZE, or the wrong IDEA?")

n = 10,142 hedged trades
  realised option P&L on lattice-predicted P&L: beta = 0.175, R^2 = 0.166
  std of prediction 6.54bp vs std of realisation 2.81bp  (ratio 0.43)

  -> the frame-frozen ratios are 5.7x too large for this payoff

=== net P&L if the basket were scaled by k (IN-SAMPLE, post-hoc) ===
    k  gross_bp  net_1x_bp  per_trade_net  std_per_trade
0.000       0.0   -17846.5         -1.760           2.81
0.250       0.0   -22731.9         -2.241           2.61
0.175       0.0   -21261.4         -2.096           2.57
0.500       0.0   -27617.2         -2.723           3.33
1.000      -0.0   -37387.9         -3.686           5.98

This is a post-hoc, in-sample rescaling — it cannot certify a strategy. It answers one question only: is the linear leg the wrong SIZE, or the wrong IDEA?


The same rescaling applied to the winner alone is the sharpest test of what
its headline gross actually was. If the hedge were replicating, shrinking it
to its statistically correct size would cost a little gross and save a lot of
bill. If the hedge was a directional accident, shrinking it destroys the
gross — because the gross *was* the accident.

In [11]:
if len(tr) and tr["lin_contracts"].max() > 0:
    rows = []
    for k, label in ((1.0, "as traded"), (beta, "beta-sized"),
                     (0.5, "half"), (0.0, "unhedged")):
        gross = tr["opt_gross_bp"] + k * tr["hedge_bp"]
        cost = tr["opt_cost_bp"] + k * tr["lin_cost_bp"]
        rows.append({"k": round(k, 3), "what": label,
                     "gross_bp": round(float(gross.sum()), 1),
                     "cost_bp": round(float(cost.sum()), 1),
                     "net_1x_bp": round(float((gross - cost).sum()), 1),
                     "per_trade_gross": round(float(gross.mean()), 3),
                     "std_per_trade": round(float(gross.std()), 2)})
    print("=== the winner, with the basket scaled (IN-SAMPLE) ===")
    print(pd.DataFrame(rows).to_string(index=False))
    print("\nA hedge shrunk to its correct size taking the P&L with it is the "
          "definition of a directional position mislabelled as a hedge.")

=== the winner, with the basket scaled (IN-SAMPLE) ===
    k       what  gross_bp  cost_bp  net_1x_bp  per_trade_gross  std_per_trade
1.000  as traded     143.8    144.9       -1.1            3.783           8.51
0.175 beta-sized      43.7     78.6      -34.9            1.149           2.92
0.500       half      83.1    104.7      -21.6            2.187           4.38
0.000   unhedged      22.5     64.5      -42.0            0.591           3.27

A hedge shrunk to its correct size taking the P&L with it is the definition of a directional position mislabelled as a hedge.


There are two candidate explanations for a beta well under one, and they have
opposite implications. **Staleness**: the ratios are frozen at entry, so a
large move over a long hold walks the package away from the frame they were
computed in — a fixable execution problem (rebalance more). **Geometry**: a
butterfly's lattice sensitivity is a density and the market's density is the
lattice's smeared by the off-lattice premium — an unfixable model problem.

They separate cleanly: if staleness dominates, beta rises toward 1 for short
holds and small realised moves; if geometry dominates, beta is flat.

In [12]:
if len(cal) > 30:
    c = cal.copy()
    c["hold_days"] = (c["exit"] - c["entry"]).dt.days
    c["pred"] = -c["hedge_bp"]
    c["move"] = c["pred"].abs()
    rows = []
    for label, grp in (("hold <= 7d", c[c["hold_days"] <= 7]),
                       ("hold 8-16d", c[(c["hold_days"] > 7)
                                        & (c["hold_days"] <= 16)]),
                       ("hold > 16d", c[c["hold_days"] > 16]),
                       ("|predicted| < 2bp", c[c["move"] < 2]),
                       ("|predicted| 2-8bp", c[(c["move"] >= 2)
                                               & (c["move"] < 8)]),
                       ("|predicted| >= 8bp", c[c["move"] >= 8])):
        if len(grp) < 30:
            continue
        p, y = grp["pred"].to_numpy(), grp["opt_gross_bp"].to_numpy()
        b = float(np.dot(p, y) / np.dot(p, p))
        rows.append({"subset": label, "n": len(grp), "beta": round(b, 3),
                     "median_hold_d": int(grp["hold_days"].median()),
                     "median_|pred|_bp": round(float(grp["move"].median()), 2)})
    print("=== hedge-ratio beta by holding period and by realised move ===")
    print(pd.DataFrame(rows).to_string(index=False))
    print("\nA beta that does not climb toward 1 for short holds and small "
          "moves is not a staleness problem.")

=== hedge-ratio beta by holding period and by realised move ===
            subset    n  beta  median_hold_d  median_|pred|_bp
        hold <= 7d 3162 0.134              5              1.69
        hold 8-16d 1102 0.243             11              1.81
        hold > 16d 5878 0.181             21              2.61
 |predicted| < 2bp 4844 0.484             20              0.77
 |predicted| 2-8bp 4136 0.413             21              3.59
|predicted| >= 8bp 1162 0.120             21             11.80

A beta that does not climb toward 1 for short holds and small moves is not a staleness problem.


### The same claim, checked without the hedge

The regression above is downstream of the hedge ratios, the jump panels and
the trade selection. If the explanation is right — the lattice's density is
sharper than the market's — it should be visible far upstream of all of that,
in the raw panel: a FIXED butterfly's lattice-fair price should simply move
more from day to day than its market price does. No hedge, no trades, no
jumps; just two price series for the same package.

In [13]:
cells = pd.read_parquet(DATA / "cells.parquet")
cells["as_of"] = pd.to_datetime(cells["as_of"])
cells = cells[cells["p_lattice"] >= 0.01].sort_values(
    ["symbol", "center_px", "as_of"])
rows = []
for (sym, k), g in cells.groupby(["symbol", "center_px"]):
    if len(g) < 15:
        continue
    g = g.set_index("as_of")
    jj = pd.concat([g["mkt_bp"].diff().rename("mkt"),
                    g["fair_bp"].diff().rename("fair")], axis=1).dropna()
    if len(jj) < 15:
        continue
    rows.append({"symbol": sym, "k": k, "n": len(jj),
                 "std_mkt": jj["mkt"].std(), "std_fair": jj["fair"].std(),
                 "corr": jj["mkt"].corr(jj["fair"]),
                 "beta": float(np.dot(jj["fair"], jj["mkt"])
                               / np.dot(jj["fair"], jj["fair"]))})
dz = pd.DataFrame(rows)
if len(dz):
    print(f"fixed-strike butterfly series compared: {len(dz)}")
    print(f"  median daily std, MARKET  price change : "
          f"{dz['std_mkt'].median():.3f}bp")
    print(f"  median daily std, LATTICE price change : "
          f"{dz['std_fair'].median():.3f}bp")
    print(f"  median ratio market / lattice          : "
          f"{(dz['std_mkt'] / dz['std_fair']).median():.3f}")
    print(f"  median beta of market change on lattice change: "
          f"{dz['beta'].median():.3f}  (corr {dz['corr'].median():.3f})")
    print(f"  share of series where the LATTICE moves more: "
          f"{(dz['std_fair'] > dz['std_mkt']).mean():.1%}")
    m = pd.cut(cells["p_lattice"], [0.01, 0.1, 0.3, 0.6, 1.0])
    mass = (cells.assign(bin=m).groupby(["symbol", "center_px"], observed=True)
            ["p_lattice"].mean().rename("mass").reset_index()
            .rename(columns={"center_px": "k"}))
    dm = dz.merge(mass, on=["symbol", "k"], how="inner")
    dm["mass_bin"] = pd.cut(dm["mass"], [0.01, 0.1, 0.3, 0.6, 1.0])
    print("\n=== by the cell's lattice mass (is it a wing artefact?) ===")
    print(dm.groupby("mass_bin", observed=True).agg(
        series=("beta", "size"), beta=("beta", "median"),
        ratio=("std_mkt", "median")).round(3).to_string())
    print("\nThis beta is computed from the panel alone — no hedge ratios, no "
          "jump marks, no trade selection — and lands in the same place as the "
          "hedge regression. Two independent routes to the same number.")

fixed-strike butterfly series compared: 120
  median daily std, MARKET  price change : 0.586bp
  median daily std, LATTICE price change : 1.065bp
  median ratio market / lattice          : 0.544
  median beta of market change on lattice change: 0.295  (corr 0.541)
  share of series where the LATTICE moves more: 86.7%

=== by the cell's lattice mass (is it a wing artefact?) ===
             series   beta  ratio
mass_bin                         
(0.01, 0.1]      18  0.376  0.443
(0.1, 0.3]       46  0.222  0.591
(0.3, 0.6]       46  0.305  0.581
(0.6, 1.0]       10  0.335  0.722

This beta is computed from the panel alone — no hedge ratios, no jump marks, no trade selection — and lands in the same place as the hedge regression. Two independent routes to the same number.


## 6. The pre-declared kill criteria, answered

In [14]:
fade = live[live["direction"] == "fade"]
med_gross_none = float(fade[fade["linear"] == "none"]["gross_bp"].median())
med_bill = float(fade[fade["linear"] == "zq"]["lin_cost_bp"].median())
k1 = med_bill > 0.5 * abs(med_gross_none)
k2 = float(live["n_trades"].median()) <= 5
# placebo retention is read on the BEST n-floored row in each world, not on the
# winner's own coordinates: a placebo world gets the same 360 trials, so the
# honest comparison is best-of-360 against best-of-360
best_by_world = {}
for world, g in league.groupby("world"):
    gl = g[g["n_trades"] > 0]
    if len(gl):
        best_by_world[world] = float(pick_winner(gl)["gross_bp"])
base_gross = best_by_world.get("real", np.nan)
placebo_keep = np.nan
if np.isfinite(base_gross) and base_gross != 0:
    others = [v for k, v in best_by_world.items() if k != "real"]
    placebo_keep = max(others) / base_gross if others else np.nan
k3 = bool(np.isfinite(placebo_keep) and placebo_keep > 0.5)
raw_best = float(live[live["expression"] == "pair_raw"]["net_1x_bp"].max())
odd_best = float(live[live["expression"].isin(
    ["pair_odd", "pair_odd_dev"])]["net_1x_bp"].max())
k4 = odd_best <= raw_best
dsr = deflated_for_grid(d, real, sharpe_col="sharpe")
k5 = (int(w["n_trades"]) >= 10 and float(w["net_2x_bp"]) > 0
      and float(dsr["dsr_prob"]) > 0.5
      and float(live["net_1x_bp"].median()) >= 0)

print("1. linear leg costs > half the gross of the paired expression?")
print(f"     median unhedged FADE gross {med_gross_none:+.1f}bp, median ZQ bill "
      f"{med_bill:.1f}bp -> {'FIRES' if k1 else 'does not fire'}")
print("2. collapses to 1-5 trades / 2y like channel-1?")
print(f"     median config runs {live['n_trades'].median():.0f} trades "
      f"-> {'FIRES' if k2 else 'does not fire'}")
print("3. placebos retain the edge?")
print(f"     best-of-360 gross by world: "
      + ", ".join(f"{k} {v:+.1f}bp" for k, v in best_by_world.items()))
print(f"     best placebo retains {placebo_keep:.0%} of the real gross "
      f"-> {'FIRES' if k3 else 'does not fire'}")
print("4. the odd rungs fail to separate from pair_raw?")
print(f"     best raw {raw_best:+.1f}bp vs best odd {odd_best:+.1f}bp "
      f"-> {'FIRES' if k4 else 'does not fire'}")
print("5. ALIVE (n>=10, positive at 2x, DSR>0.5, median config >= 0)?")
print(f"     n={int(w['n_trades'])}, net2x {w['net_2x_bp']:+.1f}, "
      f"DSR {dsr['dsr_prob']:.3f}, median config "
      f"{live['net_1x_bp'].median():+.1f} -> {'YES' if k5 else 'NO'}")

1. linear leg costs > half the gross of the paired expression?
     median unhedged FADE gross +13.6bp, median ZQ bill 95.3bp -> FIRES
2. collapses to 1-5 trades / 2y like channel-1?
     median config runs 47 trades -> does not fire
3. placebos retain the edge?
     best-of-360 gross by world: p1_gauss +49.0bp, p2_calendar +25.9bp, real +143.8bp
     best placebo retains 34% of the real gross -> does not fire
4. the odd rungs fail to separate from pair_raw?
     best raw -52.8bp vs best odd -1.1bp -> does not fire
5. ALIVE (n>=10, positive at 2x, DSR>0.5, median config >= 0)?
     n=38, net2x -146.1, DSR 0.000, median config -132.5 -> NO
